## ELT: Registro de Atendimentos da Defesa Civil do Recife

### Importação das bibliotecas

In [6]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

### Conexão com o banco
As credenciais do PostgreSQL são armazenadas no arquivo **.env**, fora do código. Antes da carga, uma conexão de teste confirma que o banco está disponível.

In [7]:
# lê as credenciais do arquivo .env
load_dotenv("../.env")

usuario = os.getenv("DB_USER")
senha = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
porta = os.getenv("DB_PORT")
banco = os.getenv("DB_NAME")

database_url = f"postgresql+psycopg2://{usuario}:{senha}@{host}:{porta}/{banco}"
engine = create_engine(database_url)

# testa a conexão (se falhar, interrompe a execução do código)
with engine.connect():
    print("Conexão criada com sucesso.")

Conexão criada com sucesso.


### Extração dos dados

In [8]:
dfs_brutos = {
    "raw_atendimentos_2022": pd.read_csv("data/atendimentos-2022.csv", sep=";", dtype=str, encoding="utf-8"),
    "raw_atendimentos_2023": pd.read_csv("data/atendimentos-2023.csv", sep=";", dtype=str, encoding="utf-8"),
    "raw_atendimentos_2024": pd.read_csv("data/atendimentos-2024.csv", sep=";", dtype=str, encoding="utf-8")
}

for nome_tabela, df_raw in dfs_brutos.items():
    print(f"{nome_tabela}: {df_raw.shape}")

raw_atendimentos_2022: (90425, 16)
raw_atendimentos_2023: (83164, 16)
raw_atendimentos_2024: (80173, 16)


### Carregamento dos dados
As bases são carregadas no PostgreSQL como tabelas brutas, identificadas pelo prefixo *raw*. Depois, a contagem de linhas de cada tabela no banco é comparada com a das bases lidas.

#### Carga das tabelas brutas

In [ ]:
for nome_tabela, df_raw in dfs_brutos.items():
    df_raw.to_sql(
        nome_tabela,
        con=engine,
        if_exists="replace", # substitui a tabela se ela já existir
        index=False # não cria a coluna de índice no banco de dados
    )

    print(f"{nome_tabela} carregada.")

raw_atendimentos_2022 carregada.
raw_atendimentos_2023 carregada.
raw_atendimentos_2024 carregada.


#### Conferência no banco

In [ ]:
for nome_tabela in dfs_brutos:
    total = pd.read_sql(
        f"SELECT COUNT(*) AS total FROM {nome_tabela}",
        con=engine
    )["total"][0] # extrai o valor da contagem que vem num dataframe 1x1

    print(f"{nome_tabela}: {total} linhas no banco.")

raw_atendimentos_2022: 90425 linhas no banco.
raw_atendimentos_2023: 83164 linhas no banco.
raw_atendimentos_2024: 80173 linhas no banco.
